# هل يمكننا إنشاء ناقل النص الخاص بنا؟



بينما كنت أعمل مع مشروع Medium، واجهت مشكلة نقص الذاكرة عندما حاولت إنشاء 1-4 نجرام. كان من المثير للاهتمام أن نرى كيف يمكن لـ ngramms الأكبر أن تعمل. لكن جهاز الكمبيوتر الخاص بي (ذاكرة الوصول العشوائي بسعة 32 جيجابايت) لا يمكنه تنفيذ مهمة إنشاء 1-4 نجرام باستخدام CounterVectorizer.



لذلك كنت أتساءل كيف يمكنني إنشاء نجرامز (تقريبًا) بأي طول وأي حجم أو هل يمكننا تخصيص أي جزء من الجملة أو النص أو المقالة يمكننا اختياره لإنشاء مجموعة من الكلمات وما إلى ذلك.



 لقد قررت استخدام القواميس كمخزن مؤقت وسيط بين ngramms بأي حجم والمصفوفة المتفرقة. دعونا نرى كيف يعمل. أولاً، نقوم بإنشاء قائمة ألعاب مكونة من جملتين للتحقق مما إذا كان البرنامج النصي الخاص بنا يعمل بشكل صحيح.	


In [ ]:
import itertools
import os
import pickle
import random
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy import sparse

PATH = "/any_path_to_data_folder/"

In [ ]:
texts = [
    "joe lives in the center of london since his birth",
    "dann loves his job because his office is right in the center of new york",
]


يتيح لك إنشاء قواميس تحتوي على كلمات فريدة لكل نص في النصوص. سنحتاج إلى المتغير "ngramm" الذي سيحدد عدد الكلمات في العبارة. على سبيل المثال، "ngramm = 2" سيعطينا مثل هذه العبارات من الجملة الأولى: 'joe'، 'joe Lives'، 'lives'، 'lives in'، 'in'، 'in the'، إلخ. وباستخدام "ngramm = 3" سنتلقى مثل هذه المجموعات: 'joe'، 'joe Lives'، 'joe Lives in'، 'lives'، 'lives in'، 'lives in the'، إلخ. كما سنضيف المتغير "ntop" الذي سنستخدمه لاحقًا.


In [ ]:
ngramm = 2
ntop = 100
word_stats = defaultdict(int)
for i in range(len(texts)):
    word_stats[i] = defaultdict(int)  # create sub-dictionary for current text
    text = texts[i]
    words = text.split()  # split the text by every word.
    for n in range(ngramm):
        phrases = [
            " ".join(word for word in words[i : i + (n + 1)])
            for i in range(len(words) - n)
        ]
        print(
            phrases
        )  # as we see we`re getting the desired result: the list of n-words phrases
        for phrase in phrases:
            word_stats[i][
                phrase
            ] += 1  # count all phrases in current text and add them to appropriate dictionary
print("************ dictionary with counted phrases ************")
print(word_stats)

لاستخدام كلماتنا وعباراتنا كحقيبة كلمات في نماذج الانحدار، يتعين علينا تحويل قاموسنا إلى مصفوفة متفرقة. بالنسبة لأولئك الذين ليسوا على دراية بهذا النوع من المصفوفات، أوصي بقراءة هذا المقال: . ولكن على أي حال، سوف نذكر أنفسنا لماذا نستخدم مصفوفة متفرقة بدلاً من كثيفة خاصة مع البيانات الكبيرة. لنقم بإنشاء إطار بيانات حيث تكون الصفوف هي النصوص والأعمدة - والعبارات هي الميزات. القيم في الخلايا - عدد ظهور كل عبارة في نص معين. ولكن في البداية يتعين علينا تحويل جميع القواميس الفرعية لكل نص إلى قاموس واحد يحتوي على عبارات فريدة ومجموع دقة هذه القواميس في جميع النصوص. النهج الحالي (باستخدام القوائم) ليس الأمثل ولكن يمكننا مقارنة أوقات التنفيذ لاحقًا.


In [ ]:
unique_ngrams = defaultdict(int)
for i in range(len(texts)):
    cur_dic = word_stats[i]
    for phrase in cur_dic.keys():
        unique_ngrams[phrase] += cur_dic[phrase]
print(unique_ngrams)  # check all records in unique dictionary

df_feat_col = list(unique_ngrams.keys())
df_feat_values = []
for i in range(len(texts)):
    cur_text_values = []
    for col in df_feat_col:
        cur_text_values.append(word_stats[i][col])
    df_feat_values.append(cur_text_values)

df_feat = pd.DataFrame(df_feat_values, columns=unique_ngrams.keys())
print(df_feat[["his", "center", "job", "york", "in"]])


في الوقت الحالي، يعمل البرنامج النصي الخاص بنا بشكل مثالي ويمكننا الآن إنشاء جدول الميزات. ولكن ماذا عن استخدام الذاكرة؟ للتحقق من السيناريو الخاص بنا، سنقوم بتحميل قاعدة بيانات تحتوي على مراجعات الأفلام وإنشاء وظيفة لمزيد من الراحة.


In [ ]:
# Your can find data by this link https://drive.google.com/file/d/1zvCa27XOuLyGAzYOeLGHfaccmK2llkca/view?usp=sharing
with open(PATH + "reviews", "rb") as fb:
    new_texts = pickle.load(fb)
print(len(new_texts))
# we`l use 1/10 of all reviews or 1250 records.
new_texts = [re.sub(r"[^\w\s]", "", str(x).lower()) for x in new_texts[:1250]]

In [ ]:
def create_text_dicts(texts, ngram=2):
    word_stats = defaultdict(int)
    for i in range(len(texts)):
        word_stats[i] = defaultdict(int)
        text = texts[i]
        words = text.split()
        for n in range(ngram):
            phrases = [
                " ".join(word for word in words[i : i + (n + 1)])
                for i in range(len(words) - n)
            ]
            for phrase in phrases:
                word_stats[i][phrase] += 1
    return word_stats


def create_unique_dict(word_stats, texts):
    unique_ngrams = defaultdict(int)
    for i in range(len(texts)):
        cur_dic = word_stats[i]
        for phrase in cur_dic.keys():
            unique_ngrams[phrase] += cur_dic[phrase]

    return unique_ngrams


def create_feat_values(word_stats, unique_ngrams, texts):
    df_feat_col = list(unique_ngrams.keys())
    df_feat_values = []
    for i in range(len(texts)):
        cur_text_values = [word_stats[i][col] for col in df_feat_col]
        df_feat_values.append(cur_text_values)

    return df_feat_col, df_feat_values

In [ ]:
%%time
word_stats = create_text_dicts(new_texts, ngram=2)
unique_ngrams = create_unique_dict(word_stats, new_texts)
features, feat_values = create_feat_values(word_stats, unique_ngrams, new_texts)
print(len(features), np.asarray(feat_values).shape)


فيما يلي لقطة شاشة لاستخدام الذاكرة على جهاز الكمبيوتر الخاص بي.
<img src="@@KEEP_00001@@> 



كما يمكننا أن نرى النهج الحالي للحصول على مصفوفة الميزات يسبب نقص الذاكرة عندما تكون هناك قاعدة بيانات كبيرة. يستهلك 1/10 من قاعدة البيانات حوالي 1/4 من الذاكرة المتوفرة. دعونا ننظر إلى المصفوفة الخاصة بنا كإطار بيانات ونحسب تناثرها أو بمعنى آخر النسبة المئوية للخلايا التي إما غير مملوءة بالبيانات أو هي أصفار.


In [ ]:
feat_table = pd.DataFrame(feat_values, columns=features)
print(feat_table.iloc[:5, :10])  # a part of matrix to avoid crash

count_zeros = feat_table.isin([0.0]).sum(
    axis=0
)  # count how many times zeros are present in each feature
print("**********")
print(count_zeros.head())

sparsity = np.sum(count_zeros) / (len(features) * len(feat_values)) * 100
print("**********")
print("Sparsity: " + str(np.round(sparsity, 2)))


كما نرى، يتكون جدول الميزات الخاص بنا من 100% تقريبًا من الأصفار وكل هذه المعلومات غير المفيدة تشغل مساحة كبيرة من الذاكرة. يمكننا إصلاح ذلك عن طريق تحويل المصفوفة الكثيفة إلى متفرقة ومقارنة الذاكرة المستخدمة بعد ذلك.


In [ ]:
sparse_feat_matrix = sparse.csr_matrix(feat_table.values)
print("Dense matrix memory: " + str(feat_table.values.nbytes))
print("Sparse matrix memory: " + str(sparse_feat_matrix.data.nbytes))

لقد قمنا بتخفيض 99٪ من الذاكرة التي كانت تستخدمها المصفوفة الكثيفة. ولكن لا يزال البرنامج النصي الخاص بنا غير فعال، لأننا قمنا بإنشاء مصفوفة كثيفة ذات ميزات كاملة في البداية وبعد ذلك فقط نقوم بتحويلها إلى مصفوفة متفرقة. حتى نتمكن من تغيير الكود بالطريقة التي يتم بها إنشاء مصفوفة متفرقة لكل نص (التكرار)، ثم يتم ربط المصفوفة المتفرقة الصغيرة في مصفوفة ميزات متفرقة كاملة.


In [ ]:
def create_feat_values(word_stats, unique_ngrams, texts):
    feat_col = list(unique_ngrams.keys())

    for i in range(len(texts)):
        cur_text_values = [word_stats[i][col] for col in df_feat_col]
        cur_text_values = np.asarray(cur_text_values)
        if i == 0:
            # convert current text dense matrix to sparse
            texts_values = sparse.csr_matrix(cur_text_values)
        else:
            cur_text_values = sparse.csr_matrix(cur_text_values)
            # concatenate cur sparse matrix into whole sparse feature matrix
            texts_values = sparse.vstack([texts_values, cur_text_values])

    return feat_col, texts_values


دعونا نتحقق مما إذا كانت المصفوفة المتفرقة تحتوي على القيم الصحيحة باستخدام مثال لعبتنا.


In [ ]:
%%time
word_stats = create_text_dicts(texts, ngram=2)
unique_ngrams = create_unique_dict(word_stats, texts)
features, feat_values = create_feat_values(word_stats, unique_ngrams, texts)
print(features)
print(len(features))
print(feat_values)


دعونا نتحقق من كلمة "له": فقد ظهرت مرة واحدة في النص الأول (0 فهرس للنصوص)، ومرتين في النص الثاني (1 فهرس). في قائمة الميزات تقف هذه الكلمة عند 8 فهرس (9 على التوالي). وإذا نظرنا إلى المصفوفة المتفرقة نجد أن الرقم الخاص بنص الفهرس 0 وكلمات الفهرس 8 (0,8) يساوي 1، وبالنسبة لنص الفهرس 1 وكلمات الفهرس 8 (1,8) يوجد الرقم 2. كما نرى أن طول القائمة ذات الميزات يساوي 39 وفهرس الكلمة الأخيرة في المصفوفة المتفرقة هو 38 وهما متساويان.


ما هي الفوائد التي نحصل عليها باستخدام هذه الطريقة لإنشاء مصفوفة متفرقة؟ بادئ ذي بدء، يمكننا إنشاء مصفوفة متفرقة بأي طول تقريبًا من n-grams. قد يؤدي استخدام ناقلات sklearn ذات ngrams الطويلة (أكثر من 4) إلى "تحميل زائد" على ذاكرة نظامك. بالطبع، يمكن أن تكون طريقة التوجيه هذه أبطأ بكثير من طريقة sklearn، ولكن عندما تكون هناك حاجة للحصول على 5 أو 6 نجرام طول - لا يهم الوقت. والفائدة الثانية التي لا تقل فائدة هي إمكانية تخصيص الناقل لاحتياجات محددة. يمكننا تحديد أطوال معينة من العبارات (ngram) التي نريد الحصول عليها: 1،2،3،5،10. أو على سبيل المثال، نحتاج إلى الحصول على عبارات بحجم 2 جرام، ليس مع الكلمات التي تقف واحدة تلو الأخرى، ولكن مع كل كلمة ثانية (تخطي الكلمة بين 1 و3، 2 و4، وما إلى ذلك): "joe in"، و"lives the"، و"in center" وما إلى ذلك. موقف آخر: لدينا قاعدة بيانات تحتوي على مراجعات الأفلام و5 علامات للتصنيف: فيلم كوميدي، ومحقق، ورعب، وكرتون، وفيلم أكشن. اهتمامنا هو الحصول على 1-5 عبارات n-gram باستخدام 100000 كلمة الأكثر شيوعًا في جميع المراجعات، ولكن 20000 كلمة الأكثر شيوعًا لكل علامة. أو علاوة على ذلك: سيعتمد عدد الكلمات العليا لكل علامة على توزيع المراجعة لكل علامة. إذا كان هناك 50000 مراجعة، و30000 منها تتعلق بالكوميديا، فسنرغب في الحصول على 60% (30000/50000) من أفضل الكلمات من مراجعات الكوميديا. دعونا ندمج وظائفنا المنفصلة في وظيفة واحدة ونضيف القليل من التخصيص للحصول على مجموعة الكلمات.


In [ ]:
def create_sparse_features(texts, target, ngram, ntop=np.inf, ntop_by_class=False):
    """
    target: list of classes according to texts. List
    
    ngram: the list of lengths for desired phrases. List  
    
    ntop: how many most popular words we want to use. Integer. Default value 
          means infiniry, that`s why the dictionary isn`t cut off by ntop value./.

    ntop_by_class: choose top words from every class according 
                   to their distribution. True or False  
    """

    word_stats = defaultdict(int)
    for i in range(len(texts)):
        word_stats[i] = defaultdict(int)
        text = texts[i]
        words = text.split()
        for n in ngram:
            # because of we changed the list of ngram from range(3) to
            # hand-input values [1,2,3] we have to change the code in this part
            phrases = [
                " ".join(word for word in words[i : i + n])
                for i in range(len(words) - n + 1)
            ]
            for phrase in phrases:
                word_stats[i][phrase] += 1

    unique_ngrams = defaultdict(int)
    if ntop_by_class:
        # count how many time each target occuries
        target_count = Counter(target)
        for key in target_count.keys():
            # count each target part in whole data
            cur_target_dist = target_count[key] / len(new_texts)
            # count how many top words we`l take from each tag texts.
            cur_target_top_words = int(cur_target_dist * ntop)
            # create empty dictionary for current target
            cur_target_ngrams = defaultdict(int)
            # create list of indexes for current target
            cur_target_ind = np.where(np.asarray(target) == key)[0]
            # using loop and list of indexes fill current target dictionary
            # from corresponding text dictionaries and count unique phrases
            for n in cur_target_ind:
                cur_dic = word_stats[n]
                for phrase in cur_dic.keys():
                    cur_target_ngrams[phrase] += cur_dic[phrase]
            # if length of current target dictionary is more than
            # cur_target_top_words value - sort dictionary by values and cut off it
            # content by that value.
            if len(cur_target_ngrams) > cur_target_top_words:
                cur_target_ngrams = dict(
                    sorted(cur_target_ngrams.items(), key=lambda kv: kv[1])
                )
                cur_target_ngrams = dict(
                    itertools.islice(
                        cur_target_ngrams.items(),
                        len(cur_target_ngrams) - cur_target_top_words,
                        len(cur_target_ngrams),
                    )
                )
            # connect current target dictionary with whole unique_ngrams dictionary
            unique_ngrams = {**unique_ngrams, **cur_target_ngrams}

    else:
        for i in range(len(texts)):
            cur_dic = word_stats[i]
            for phrase in cur_dic.keys():
                unique_ngrams[phrase] += cur_dic[phrase]
        if len(unique_ngrams) > ntop:
            unique_ngrams = dict(sorted(unique_ngrams.items(), key=lambda kv: kv[1]))
            unique_ngrams = dict(
                itertools.islice(
                    unique_ngrams.items(), len(unique_ngrams) - ntop, len(unique_ngrams)
                )
            )

    feat_col = list(unique_ngrams.keys())
    for i in range(len(texts)):
        cur_text_phares = [word_stats[i][col] for col in feat_col]
        cur_text_phares = np.asarray(cur_text_phares)
        if i == 0:
            feat_values = sparse.csr_matrix(cur_text_phares)
        else:
            cur_feat_values = sparse.csr_matrix(cur_text_phares)
            feat_values = sparse.vstack([feat_values, cur_feat_values])

    return feat_col, feat_values


لنقم بتشغيل الكود الخاص بنا مع وبدون استخدام التوزيع المستهدف لكلمات ntop.


In [ ]:
%%time
# To run the vectorizer we need tags for our review database.
tags = ["comedy", "detective", "horror", "cartoon", "action"]
target = [tags[random.randrange(len(tags))] for item in range(len(new_texts))]
ngram = [1, 2, 3, 5, 10]
features, feat_values = create_sparse_features(
    new_texts, target, ngram, ntop=5000, ntop_by_class=False
)
print(feat_values.shape)

In [ ]:
%%time
# To run the vectorizer we need tags for our review database.
tags = ["comedy", "detective", "horror", "cartoon", "action"]
target = [tags[random.randrange(len(tags))] for item in range(len(new_texts))]
ngram = [1, 2, 3, 5, 10]
features, feat_values = create_sparse_features(
    new_texts, target, ngram, ntop=5000, ntop_by_class=True
)
print(feat_values.shape)


يكون شكل المصفوفة المتفرقة بعد التشغيل الثاني للبرنامج النصي أقل من قيمة ntop المحددة. يتحقق ذلك `s why 2/3 of most popular words in texts are intersected within the tags. But let` من استخدام الذاكرة - قم بتشغيل البرنامج النصي دون تعيين قيمة ntop وبقيم ngram كما في المصفوفة الكثيفة أعلاه.


In [ ]:
%%time
# To run the vectorizer we need tags for our review database.
tags = ["comedy", "detective", "horror", "cartoon", "action"]
target = [tags[random.randrange(len(tags))] for item in range(len(new_texts))]
ngram = [1, 2]
features, feat_values = create_sparse_features(new_texts, target, ngram)
print(feat_values.shape)

إليكم لقطة الشاشة الثانية لاستخدام الذاكرة على جهاز الكمبيوتر الخاص بي.
<img src="@@KEEP_00002@@> 



لسوء الحظ، فإن فكرتنا المتمثلة في إنشاء مصفوفة متفرقة أثناء التكرار والجمع بينها في مصفوفة كبيرة لا تعمل بشكل صحيح. كما نرى، فإن استخدام الذاكرة هو نفسه كما هو الحال مع المصفوفة الكثيفة أثناء التكرار، وفقط بعد ذلك يتم قطع الذاكرة إلى حجم المصفوفة المتفرقة النهائية. ماذا يمكننا أن نفعل بهذا؟ دعونا نحاول تحسين البرنامج النصي لدينا. في البداية، لن نستخدم قواميس كل نص أثناء إنشاء مصفوفة متفرقة. بدلاً من ذلك، سنقوم بإنشاء عبارات مرة أخرى، والتحقق مما إذا كانت موجودة في قاموس فريد، وحساب مدى دقتها وبعد ذلك إنشاء مصفوفة متفرقة باستخدام مصفوفة الأصفار غير المكتملة. في المرحلة الثانية، سنقوم بحفظ كل مصفوفة متفرقة على القرص الصلب ثم نقرأها ونجمعها في مصفوفة واحدة.


In [ ]:
def create_sparse_features(texts, target, ngram, ntop=np.inf, ntop_by_class=False):
    """
    target: list of classes according to texts. List
    
    ngram: the list of lengths for desired phrases. List  
    
    ntop: how many most popular words we want to use. Integer. Default value 
          means infiniry, that`s why the dictionary isn`t cut off by ntop value./. 

    ntop_by_class: choose top words from every class according 
                   to their distribution. True or False  
    """

    word_stats = defaultdict(int)
    for i in range(len(texts)):
        word_stats[i] = defaultdict(int)
        text = texts[i]
        words = text.split()
        for n in ngram:
            phrases = [
                " ".join(word for word in words[i : i + n])
                for i in range(len(words) - n + 1)
            ]
            for phrase in phrases:
                word_stats[i][phrase] += 1

    unique_ngrams = defaultdict(int)
    if ntop_by_class:

        target_count = Counter(target)
        for key in target_count.keys():
            cur_target_dist = target_count[key] / len(new_texts)
            cur_target_top_words = int(cur_target_dist * ntop)
            cur_target_ngrams = defaultdict(int)
            cur_target_ind = np.where(np.asarray(target) == key)[0]

            for n in cur_target_ind:
                cur_dic = word_stats[n]
                for phrase in cur_dic.keys():
                    cur_target_ngrams[phrase] += cur_dic[phrase]

            if len(cur_target_ngrams) > cur_target_top_words:
                cur_target_ngrams = dict(
                    sorted(cur_target_ngrams.items(), key=lambda kv: kv[1])
                )
                cur_target_ngrams = dict(
                    itertools.islice(
                        cur_target_ngrams.items(),
                        len(cur_target_ngrams) - cur_target_top_words,
                        len(cur_target_ngrams),
                    )
                )
            unique_ngrams = {**unique_ngrams, **cur_target_ngrams}

    else:
        for i in range(len(texts)):
            cur_dic = word_stats[i]
            for phrase in cur_dic.keys():
                unique_ngrams[phrase] += cur_dic[phrase]
        if len(unique_ngrams) > ntop:
            unique_ngrams = dict(sorted(unique_ngrams.items(), key=lambda kv: kv[1]))
            unique_ngrams = dict(
                itertools.islice(
                    unique_ngrams.items(), len(unique_ngrams) - ntop, len(unique_ngrams)
                )
            )
    # create dictionary with features indexes
    all_phrases = unique_ngrams.keys()
    all_phrases_dic_ind = defaultdict(int)
    for key in unique_ngrams.keys():
        all_phrases_dic_ind[key] = len(all_phrases_dic_ind)

    # create sparse matrix to each text
    for i in range(len(texts)):
        # create proper name for right sorting files according to features index
        name = "0" * (len(str(len(texts))) - len(str(i))) + str(i)
        text = texts[i]
        # create from current text list of words
        words = text.split()
        # create list of ngram phrases dropping those that are not in unique
        # dictionary
        cur_phrases = []
        for ii in range(len(words)):
            for n in ngram:
                phrase = " ".join(word for word in words[ii : ii + n])
                if phrase in all_phrases:
                    cur_phrases.append(phrase)
        # count accurancies of chosen phrases in the current text
        cur_phrases_count = Counter(cur_phrases)
        # create 1 row dense matrix using numpy zeroes matrix, where the number
        # of columns equals the length of unique phrases dictionary
        dense_matrix = np.zeros([1, len(all_phrases)])
        for key in cur_phrases_count.keys():
            # get the index of current feature of current text from unique index
            # dictionary
            all_phrases_ind = all_phrases_dic_ind[key]
            # put the number of accurance of phrase to the dense matrix at certain index
            # according to the index of phrase in the unique index dictionary
            dense_matrix[:, all_phrases_ind] = cur_phrases_count[key]
        # create and save sparse matrix
        csr_matrix = sparse.csr_matrix(dense_matrix)
        sparse.save_npz(PATH + "/sparses/" + str(name), csr_matrix)

    # laod sparse matrix and create the big one
    sparse_files = os.listdir(PATH + "/sparses/")
    for i in range(len(sparse_files)):
        if i == 0:
            feat_values = sparse.load_npz(PATH + "/sparses/" + sparse_files[i])
        else:
            cur_sparse = sparse.load_npz(PATH + "/sparses/" + sparse_files[i])
            feat_values = sparse.vstack([feat_values, cur_sparse])

    return unique_ngrams.keys(), feat_values

In [ ]:
%%time
# To run the vectorizer we need tags for our review database.
tags = ["comedy", "detective", "horror", "cartoon", "action"]
target = [tags[random.randrange(len(tags))] for item in range(len(new_texts))]
ngram = [1, 2]
features, feat_values = create_sparse_features(new_texts, target, ngram)
print(feat_values.shape)


فيما يلي لقطة شاشة لاستخدام الذاكرة على جهاز الكمبيوتر الخاص بي باستخدام هذا البرنامج النصي
<img src="@@KEEP_00003@@> 



كما نرى، حصلنا على نفس شكل مصفوفة الميزات مثل الكثافة، ولكن أسرع بخمس مرات وبدون استخدام أي ذاكرة تقريبًا. لقد جربت هذا الأسلوب بإعدادات مختلفة وتمكنت من الحصول على مصفوفة بأشكال تصل إلى 70000 صف و20 مليون ميزة كأعمدة دون استخدام جدي للذاكرة. وبصرف النظر عن هذا، يمكن تعديل هذا النص لأي احتياجات لإنشاء أي بنية لمجموعة معينة من الكلمات. 
